## Helper classes for double stage Monte Carlo analysis

Create a `DoubleStageRocket` with all parameters of full Dauntless model (sustainer motor should not be added)

Create a `Rocket` for the sustainer and another for the booster (use `EmptyMotor` or a motor with negligible thrust and burn time)

Add either the sustainer or the booster (only one for a Monte Carlo analysis)

Create stochastic objects as usual (replace the `Rocket` for `DoubleStageRocket`)

In [ ]:
from rocketpy import Rocket

class DoubleStageRocket(Rocket):
    def __init__(
        self,
        name,
        radius,
        mass,
        inertia,
        power_off_drag,
        power_on_drag,
        center_of_mass_without_motor,
        coordinate_system_orientation="tail_to_nose",
    ):
        super().__init__(
            name,
            radius,
            mass,
            inertia,
            power_off_drag,
            power_on_drag,
            center_of_mass_without_motor,
            coordinate_system_orientation,
        )
        self.child_stage = None

    def add_child_stage(self, child_stage):
        self.child_stage = child_stage

In [ ]:
from rocketpy import MonteCarlo
from rocketpy import Flight

class DoubleStageMonteCarlo(MonteCarlo):
    def __init__(
        self,
        filename,
        environment,
        rocket,
        flight,
        export_list=None,
        data_collector=None,
    ):
        super().__init__(
            filename,
            environment,
            rocket,
            flight,
            export_list,
            data_collector,
        )

    def __run_single_simulation(self):
        env = self.environment.create_object()
        if self.rocket.child_stage is not None:
            first_stage = Flight(
                rocket=self.rocket.create_object(),
                environment=env,
                rail_length=self.flight._randomize_rail_length(),
                inclination=self.flight._randomize_inclination(),
                heading=self.flight._randomize_heading(),
                initial_solution=self.flight.initial_solution,
                terminate_on_apogee=True,
                time_overshoot=self.flight.time_overshoot,
            )
            return Flight(
                rocket=self.rocket.child_stage.create_object(),
                environment=env,
                rail_length=0,
                inclination=0,
                heading=0,
                terminate_on_apogee=self.flight.terminate_on_apogee,
                time_overshoot=self.flight.time_overshoot,
                initial_solution=first_stage,
            )
        return Flight(
            rocket=self.rocket.create_object(),
            environment=env,
            rail_length=self.flight._randomize_rail_length(),
            inclination=self.flight._randomize_inclination(),
            heading=self.flight._randomize_heading(),
            initial_solution=self.flight.initial_solution,
            terminate_on_apogee=self.flight.terminate_on_apogee,
            time_overshoot=self.flight.time_overshoot,
        )